# Causal-Inference Enhanced Actor-Critic SQL Generation

This notebook demonstrates how **causal inference** makes the **Actor-Critic** design pattern more robust and less error-prone for SQL generation.

## The Problem: Confounded Feedback Loops

In a standard Actor-Critic workflow:
- The **Actor** (Generator) produces SQL from natural language
- The **Critic** (Validator) evaluates the SQL and provides feedback
- On failure, the Actor tries again with the Critic's feedback

But this loop is vulnerable to **confounding**: hidden variables (query complexity, token limits, context exhaustion) affect both the Actor's output quality AND the Critic's evaluation. The Critic cannot distinguish "Actor chose a bad strategy" from "an environmental constraint prevented success." This leads to **deadlocks** — the Actor is told to fix something that no strategy can fix.

## The Causal Fix: Deconfounding

We insert a **Constraint Collector** between Actor and Critic that makes hidden constraints observable (Pearl's backdoor adjustment). After a rejection, a **Causal Diagnosis Agent** determines whether the failure is due to:
- `ACTOR_STRATEGY_FAILURE` — fixable, re-route to Actor with targeted feedback
- `ENVIRONMENTAL_CONSTRAINT` — not fixable by Actor, accept with caveat
- `MIXED` — fix what's fixable, caveat the rest

| Component | Role |
|-----------|------|
| Actor (Generator) | Generates SQL from natural language |
| Constraint Collector | Packages environmental metadata (deconfounding layer) |
| Critic (Validator) | Evaluates SQL with constraint-awareness |
| Causal Diagnosis | Classifies rejection root cause |
| Causal Router | Routes based on causal attribution, not just verdict |

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure src/ is on the Python path
src_dir = str(Path(".").resolve().parent)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

In [ ]:
from causal_inference.settings import Settings
from causal_inference.llm_client import build_llm_client

settings = Settings.from_env()
llm = build_llm_client(settings)
print(f"LLM provider: {settings.llm_provider}")

## 2. Understanding the Constraint Collector (Deconfounding Layer)

Before running the full workflow, let's see how the Constraint Collector works. It scores query complexity and classifies the constraint regime — this metadata is what breaks the confounding cycle.

In [ ]:
from causal_actor_critic.constraint_collector import (
    compute_query_complexity,
    classify_constraint,
)

# Simple query — should be UNCONSTRAINED
simple_q = "Show the top 10 nations by total revenue."
score_s, factors_s = compute_query_complexity(simple_q)
print(f"Simple query complexity: {score_s:.3f}")
print(f"  Factors: {factors_s}")
print(f"  Constraint class: {classify_constraint(score_s, 0, 3, 200)}")
print()

# Complex query — should be COMPLEXITY_CONSTRAINED
complex_q = (
    "Show monthly revenue trends by region with year-over-year growth rates, "
    "running totals, and rank each region within its quarter using window functions."
)
score_c, factors_c = compute_query_complexity(complex_q)
print(f"Complex query complexity: {score_c:.3f}")
print(f"  Factors: {factors_c}")
print(f"  Constraint class: {classify_constraint(score_c, 0, 3, 200)}")
print()

# Same complex query but on final attempt — CONTEXT_EXHAUSTED
print(f"Same query, attempt 3/3: {classify_constraint(score_c, 3, 3, 200)}")

The constraint class tells the Critic how to adjust its rubric:

| Constraint Class | Critic Behavior |
|-----------------|----------------|
| `UNCONSTRAINED` | Full rubric — strict evaluation |
| `COMPLEXITY_CONSTRAINED` | Evaluate strategy quality, not perfection |
| `SIZE_CONSTRAINED` | Accept partial if prioritization is sound |
| `CONTEXT_EXHAUSTED` | Accept best-effort |

## 3. Build the Causal Actor-Critic Workflow

The workflow adds three components that the standard Actor-Critic lacks:
1. **Constraint Collector** between Actor and Critic
2. **Causal Diagnosis** after Critic rejection
3. **Three-way causal routing** instead of simple verdict routing

In [ ]:
from causal_actor_critic.graph import build_causal_actor_critic_workflow

graph = build_causal_actor_critic_workflow(llm)

print("Workflow nodes:")
for node in graph.get_graph().nodes:
    print(f"  • {node}")

In [ ]:
# Visualize the graph (optional — requires graphviz)
try:
    from IPython.display import Image, display
    img_data = graph.get_graph().draw_mermaid_png()
    display(Image(img_data))
except Exception as e:
    print(f"Graph visualization unavailable: {e}")
    print("The workflow runs regardless — visualization is optional.")

## 4. Example 1: Simple Query (UNCONSTRAINED)

A straightforward query where the standard Actor-Critic would also work fine. The causal enhancement adds negligible overhead here but still provides structured metadata.

In [ ]:
result_1 = await graph.ainvoke({
    "user_query": "Show the top 10 nations by total revenue from line items, ordered descending.",
})

print(f"Status: {result_1.get('status', 'N/A')}")
print(f"Attempts: {result_1.get('attempt', 'N/A')}")
print(f"Constraint class: {result_1.get('constraint_metadata', {}).get('constraint_class', 'N/A')}")
print(f"\n--- Generated SQL ---")
print(result_1.get('final_sql', result_1.get('generated_sql', 'N/A')))
print(f"\n--- Explanation ---")
print(result_1.get('final_explanation', result_1.get('sql_explanation', 'N/A')))

## 5. Example 2: Complex Analytical Query (COMPLEXITY_CONSTRAINED)

This is where the causal enhancement shines. The query requires window functions, temporal aggregation, and multi-table joins. Without causal awareness, the Critic might reject the Actor's output for being "incomplete" when the real issue is that the query is inherently complex.

With the constraint metadata, the Critic knows to evaluate **strategy quality** rather than demanding perfection on the first attempt.

In [ ]:
result_2 = await graph.ainvoke({
    "user_query": (
        "Show monthly revenue by region for 1995-1996 with year-over-year "
        "growth rates. Rank each region within each month by revenue using "
        "window functions. Include a running total of revenue per region."
    ),
})

print(f"Status: {result_2.get('status', 'N/A')}")
print(f"Attempts: {result_2.get('attempt', 'N/A')}")
print(f"Constraint class: {result_2.get('constraint_metadata', {}).get('constraint_class', 'N/A')}")
print(f"Complexity score: {result_2.get('constraint_metadata', {}).get('query_complexity', 'N/A')}")
print(f"Complexity factors: {result_2.get('constraint_metadata', {}).get('complexity_factors', 'N/A')}")
print(f"\n--- Generated SQL ---")
print(result_2.get('final_sql', result_2.get('generated_sql', 'N/A')))
print(f"\n--- Explanation ---")
print(result_2.get('final_explanation', result_2.get('sql_explanation', 'N/A')))

## 6. Example 3: Multi-dimensional Analytics (Testing Causal Diagnosis)

A deliberately challenging query that may trigger causal diagnosis. The system should identify which failures are Actor-fixable and which are constraint-driven.

In [ ]:
result_3 = await graph.ainvoke({
    "user_query": (
        "Create a comprehensive supplier performance dashboard: for each supplier, "
        "show their rank by total revenue within their nation, the percentile of "
        "their average delivery time compared to all suppliers, a year-over-year "
        "growth rate of their revenue, and a running total of orders fulfilled. "
        "Include only suppliers with above-median revenue in their region. "
        "Use window functions and CTEs."
    ),
})

print(f"Status: {result_3.get('status', 'N/A')}")
print(f"Attempts: {result_3.get('attempt', 'N/A')}")
print(f"Constraint class: {result_3.get('constraint_metadata', {}).get('constraint_class', 'N/A')}")
print(f"Complexity score: {result_3.get('constraint_metadata', {}).get('query_complexity', 'N/A')}")
print(f"\n--- Generated SQL ---")
print(result_3.get('final_sql', result_3.get('generated_sql', 'N/A')))

## 7. Inspect Causal Diagnosis and Correction History

This is the key innovation: after each Critic rejection, the Causal Diagnosis Agent classifies the root cause. Let's inspect the causal diagnosis and correction history for each example.

In [ ]:
import json

for i, result in enumerate([result_1, result_2, result_3], start=1):
    print(f"{'='*60}")
    print(f"Example {i}")
    print(f"{'='*60}")
    
    # Constraint metadata
    meta = result.get("constraint_metadata", {})
    print(f"\nConstraint Metadata:")
    if meta:
        print(json.dumps(meta, indent=2))
    else:
        print("  (none collected)")
    
    # Causal diagnosis
    diag = result.get("causal_diagnosis", {})
    print(f"\nCausal Diagnosis:")
    if diag:
        print(f"  Root cause: {diag.get('root_cause', 'N/A')}")
        if diag.get('actor_issues'):
            print(f"  Actor issues (fixable): {diag['actor_issues']}")
        if diag.get('environmental_issues'):
            print(f"  Environmental issues (not fixable): {diag['environmental_issues']}")
        print(f"  Should re-route to Actor: {diag.get('should_reroute', 'N/A')}")
        print(f"  Should relax rubric: {diag.get('should_relax_rubric', 'N/A')}")
    else:
        print("  (no diagnosis — query passed on first attempt)")
    
    # Correction history
    history = result.get("correction_history", [])
    print(f"\nCorrection History ({len(history)} entries):")
    for entry in history:
        print(f"  Attempt {entry.get('attempt', '?')}: source={entry.get('source', '?')}, "
              f"sql_length={len(entry.get('sql', ''))} chars")
    
    # Critic issues
    issues = result.get("critic_issues", [])
    if issues:
        print(f"\nCritic Issues ({len(issues)}):")
        for issue in issues:
            print(f"  [{issue.get('severity', '?')}] {issue.get('category', '?')}: {issue.get('description', '')}")
    print()

## 8. How Causal Deconfounding Prevents Deadlocks

### The Confounding Problem (Without Causal Enhancement)

```
Query Complexity (U)          ← Hidden confounder
    /              \
   ↓                ↓
Actor Output    Critic Evaluation    ← Both affected by U
   ↑                │
   └── Feedback ←───┘               ← Confounded signal!
```

The Critic sees incomplete output and attributes it to Actor failure. But the real cause is query complexity. The Actor is sent to fix something unfixable → oscillation → deadlock.

### The Causal Fix (With Enhancement)

```
Query Complexity (U)          ← Now OBSERVED via Constraint Collector
    /        |        \
   ↓         ↓         ↓
Actor    Constraint   Critic     ← Critic sees U directly
Output   Metadata     Evaluation
             ↓           │
         Causal          │
         Diagnosis ←─────┘       ← Root cause identified
             │
    ┌────────┼────────┐
    ↓        ↓        ↓
 Actor    Accept    Fix what's
 fix    with caveat  fixable
```

The backdoor path through U is **blocked** because the Critic now conditions on the constraint metadata. Feedback targets the Actor's actual strategy failures, not symptoms of hidden constraints.

## 9. Convergence Conditions

The causal-enhanced workflow satisfies three conditions required for convergence:

| Condition | Standard Actor-Critic | Causal-Enhanced |
|-----------|----------------------|----------------|
| **Feasibility** — exists a strategy the Critic will accept | Violated when constraints make rubric unreachable | Restored — Critic relaxes rubric under active constraints |
| **Observability** — Critic sees all variables affecting output | Violated — hidden constraints invisible | Restored — Constraint Collector routes metadata to Critic |
| **Monotonicity** — each attempt moves closer to acceptance | Violated — Actor oscillates between strategies | Restored — targeted feedback addresses actual failures |

## 10. Custom Query

Try your own query below:

In [ ]:
custom_query = """
For each market segment, find the top 3 customers by total order value
in 1995. Show their rank, total order value, and the percentage of
their segment's total revenue they represent.
""".strip()

result_custom = await graph.ainvoke({"user_query": custom_query})

print(f"Status: {result_custom.get('status', 'N/A')}")
print(f"Attempts: {result_custom.get('attempt', 'N/A')}")
print(f"Constraint class: {result_custom.get('constraint_metadata', {}).get('constraint_class', 'N/A')}")
print(f"\n--- Generated SQL ---")
print(result_custom.get('final_sql', result_custom.get('generated_sql', 'N/A')))
print(f"\n--- Explanation ---")
print(result_custom.get('final_explanation', result_custom.get('sql_explanation', 'N/A')))